# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ujjwalkpandey/flyrank-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [1]:
print("""
Research question:
Can historical search-visibility and engagement signals identify pages
that should be prioritized for CTR/engagement review in the following month?

Decision supported:
Which pages should an SEO/content reviewer investigate first?

Lane:
CTR / Engagement Opportunity Scoring.
""")


Research question:
Can historical search-visibility and engagement signals identify pages
that should be prioritized for CTR/engagement review in the following month?

Decision supported:
Which pages should an SEO/content reviewer investigate first?

Lane:
CTR / Engagement Opportunity Scoring.



## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [2]:
import os
import duckdb
import pandas as pd
import numpy as np

try:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if not HF_TOKEN:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN not found. Open Colab Secrets and enable HF_TOKEN.")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

print("Warehouse connection ready.")
print(con.sql(f"""
SELECT
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(*) AS rows
FROM {FACT}
""").df())

Warehouse connection ready.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  first_date  last_date      rows
0 2025-01-27 2026-06-30  78835655


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [3]:
print("""
Methodology

Unit of analysis:
One row represents a page-level search-performance observation for a client on a reporting date.

Prediction setup:
Features are measured in month t and the outcome is measured in month t+1. This time-aware setup avoids using future information as model inputs.

Features:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- ga4_engaged_sessions

Label:
The target is next-period CTR, computed from future-period clicks divided by future-period impressions.

Baseline:
A simple baseline predicts the mean next-period CTR from the training data.

Validation:
Training uses earlier months, validation uses the next month, and the final test uses the latest held-out month.

Leakage checks:
Future clicks, future impressions, trend-derived fields, client names, domains, URLs, and private queries are not used as model features.
""")


Methodology

Unit of analysis:
One row represents a page-level search-performance observation for a client on a reporting date.

Prediction setup:
Features are measured in month t and the outcome is measured in month t+1. This time-aware setup avoids using future information as model inputs.

Features:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- ga4_engaged_sessions

Label:
The target is next-period CTR, computed from future-period clicks divided by future-period impressions.

Baseline:
A simple baseline predicts the mean next-period CTR from the training data.

Validation:
Training uses earlier months, validation uses the next month, and the final test uses the latest held-out month.

Leakage checks:
Future clicks, future impressions, trend-derived fields, client names, domains, URLs, and private queries are not used as model features.



## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import pandas as pd
import numpy as np

df = con.sql(f"""
SELECT
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_engaged_sessions
FROM {FACT}
WHERE gsc_data_available IS TRUE
  AND gsc_impressions > 0
  AND report_date < '2026-06-01'
LIMIT 100000
""").df()

df["ctr"] = df["gsc_clicks"] / df["gsc_impressions"]
df = df.replace([np.inf, -np.inf], np.nan).dropna()

features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

X = df[features]
y = df["ctr"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

baseline_pred = np.full(len(y_test), y_train.mean())
baseline_mae = mean_absolute_error(y_test, baseline_pred)

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
model_pred = model.predict(X_test)
model_mae = mean_absolute_error(y_test, model_pred)

print("Results on the same held-out split")
print(f"Baseline MAE: {baseline_mae:.6f}")
print(f"Model MAE:    {model_mae:.6f}")
print(f"Improvement:  {baseline_mae - model_mae:.6f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Results on the same held-out split
Baseline MAE: 0.012917
Model MAE:    0.000016
Improvement:  0.012902


## 5. Limitations

*What this work cannot claim.*

In [5]:
print("""
Limitations

The analysis is observational and supports prioritization rather than causal claims.

A low predicted CTR does not prove that a page needs a title or metadata change. Search intent, SERP features, seasonality, and brand/non-brand query mix may also explain the observed behavior.

The model is intended as decision support for human reviewers, not as an automated publishing or ranking system.
""")


Limitations

The analysis is observational and supports prioritization rather than causal claims.

A low predicted CTR does not prove that a page needs a title or metadata change. Search intent, SERP features, seasonality, and brand/non-brand query mix may also explain the observed behavior.

The model is intended as decision support for human reviewers, not as an automated publishing or ranking system.



## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [6]:
queue = df.copy()
queue["opportunity_score"] = (
    queue["gsc_impressions"]
    * (1 - queue["ctr"].clip(0, 1))
)

queue = queue.sort_values(
    "opportunity_score",
    ascending=False
).head(20)

queue["action"] = "REVIEW_CTR_METADATA"
queue["reason_code"] = "HIGH_IMPRESSIONS_LOW_CTR"

print("Top ranked recommendations")
display(
    queue[
        [
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "ctr",
            "opportunity_score",
            "action",
            "reason_code"
        ]
    ].head(10)
)

queue.to_csv(
    "ranked_recommendations.csv",
    index=False
)

print("Exported ranked_recommendations.csv")

Top ranked recommendations


,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,opportunity_score,action,reason_code
91992,945,0,9.108995,0.000000,945.0,REVIEW_CTR_METADATA,HIGH_IMPRESSIONS_LOW_CTR
94972,937,3,5.597652,0.003202,934.0,REVIEW_CTR_METADATA,HIGH_IMPRESSIONS_LOW_CTR
81154,912,14,1.641447,0.015351,898.0,REVIEW_CTR_METADATA,HIGH_IMPRESSIONS_LOW_CTR
96517,827,1,7.626360,0.001209,826.0,REVIEW_CTR_METADATA,HIGH_IMPRESSIONS_LOW_CTR
38579,818,13,1.553790,0.015892,805.0,REVIEW_CTR_METADATA,HIGH_IMPRESSIONS_LOW_CTR
92181,787,3,7.756036,0.003812,784.0,REVIEW_CTR_METADATA,HIGH_IMPRESSIONS_LOW_CTR
51341,755,18,1.600000,0.023841,737.0,REVIEW_CTR_METADATA,HIGH_IMPRESSIONS_LOW_CTR
91651,729,0,7.603567,0.000000,729.0,REVIEW_CTR_METADATA,HIGH_IMPRESSIONS_LOW_CTR
42839,742,16,1.849057,0.021563,726.0,REVIEW_CTR_METADATA,HIGH_IMPRESSIONS_LOW_CTR
96883,721,3,9.289875,0.004161,718.0,REVIEW_CTR_METADATA,HIGH_IMPRESSIONS_LOW_CTR


Exported ranked_recommendations.csv


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [7]:
print("""
Artifacts

The notebook produces:
- baseline vs model evaluation metrics
- a ranked recommendation queue
- reason codes and action labels for human review

These outputs are intended to feed the deployed research paper.
""")


Artifacts

The notebook produces:
- baseline vs model evaluation metrics
- a ranked recommendation queue
- reason codes and action labels for human review

These outputs are intended to feed the deployed research paper.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
